<div style="background:linear-gradient(90deg,#C8102E 0%,#7A0019 100%); color:white; padding:28px 32px; border-radius:8px">
  <div style="font-size:0.95em; letter-spacing:2px; opacity:0.85">CLINICAL TRIAL PRE-SCREENING · ML SESSION · COMPLETED REFERENCE</div>
  <div style="font-size:2.3em; font-weight:700; margin-top:6px">🧬 Clinical Trial Patient Pre-Screening on OMOP</div>
  <div style="font-size:1.15em; margin-top:10px; max-width:880px; opacity:0.95">
    Find every patient who might qualify for a breast-cancer trial, including the ones
    whose biomarker status lives only in free-text pathology notes, using SQL, Lakeflow,
    Foundation Models, MLflow, and Genie on the Databricks Data Intelligence Platform.
  </div>
</div>

<div style="background:#E8F5E9; border-left:6px solid #2E7D32; padding:14px 18px; border-radius:4px">
<b>📖 This is the COMPLETED reference set.</b> Every <code>TODO</code> from the starter kit is
filled in and runnable, in presentation order. Use it to walk stakeholders through the solution and
to see the whole build end to end. The starter-kit skeletons (the ones the team builds during the
session) live one folder up in <code>notebooks/</code>; the intended answers are in
<code>reference/ANSWER_KEY.md</code>.
<br><br>
<b>The teaching arc:</b> <code>ai_query</code> over free text (nb 04) · a HuggingFace model
registered to Unity Catalog and served on an endpoint (nb 05) · MLflow evaluation with traces,
an LLM-as-judge, and a custom metric (nb 07) · Genie self-serve (nb 08) · the coordinator app
(<code>app/</code>).
</div>

## 👋 A starter kit: the team builds the core

<div style="background:#FFF8E1; border-left:6px solid #F2A900; padding:14px 18px; border-radius:4px">
The <b>plumbing is wired for you</b>: the 6 OMOP tables (from the shared foundation), the
pipeline skeleton, Unity Catalog
governance, all the boilerplate. <b>The team</b> writes the learnable logic: the eligibility SQL,
the biomarker pivot, the <code>ai_query</code> NLP extraction, and the MLflow evaluation.<br><br>
In the starter kit, look for <b><code>&#35; TODO (you build this)</code></b> markers. In this
completed set those are all filled in.
</div>

## 🩺 The business problem

A research coordinator needs to pre-screen patients for two breast-cancer trials:

| Trial | Looking for |
|---|---|
| **Trial A**, HER2+ | Breast cancer · **HER2 Positive** · age 18 to 75 · **no** prior anti-HER2 therapy |
| **Trial B**, ER+/HER2− | Breast cancer · **ER Positive** · **HER2 Negative** · **postmenopausal** · age 18 to 75 |
| **Trial C**, triple-negative | Breast cancer · **HER2 Negative** · **ER Negative** · **PR Negative** · age 18 to 75 |

📋 **The full, canonical eligibility card for all three trials (every `req_*` field, the eligible
counts, and the "one rule" for matching) lives in `SHARED_FOUNDATION.md`.** That is the single
source of truth. Do not re-derive the criteria from memory; read the card.

The catch: **biomarker status is not always in the structured tables.** For a large slice of
patients, HER2/ER/PR status was only ever written into the free-text pathology report. A SQL
query over `measurement` alone **silently misses them**, and a missed patient is a missed
chance at treatment.

## ✨ The value moment: structured query vs. NLP

Our 300 synthetic patients fall into three groups by where their biomarkers live:

<div style="display:flex; gap:14px; flex-wrap:wrap; margin-top:8px">
  <div style="flex:1; min-width:230px; background:#E8F5E9; border-radius:6px; padding:14px">
    <div style="font-size:1.6em; font-weight:700; color:#2E7D32">180</div>
    <b>both-agree</b> (person 1 to 180)<br>biomarkers in <i>both</i> structured tables and notes.
    <br>→ our NLP <b>ground truth</b>.
  </div>
  <div style="flex:1; min-width:230px; background:#FFEBEE; border-radius:6px; padding:14px">
    <div style="font-size:1.6em; font-weight:700; color:#C62828">60</div>
    <b>notes-only</b> (person 181 to 240)<br>biomarkers <i>only</i> in <code>note_text</code>.
    <br>→ <b>invisible to SQL</b>; recovered only by NLP.
  </div>
  <div style="flex:1; min-width:230px; background:#E3F2FD; border-radius:6px; padding:14px">
    <div style="font-size:1.6em; font-weight:700; color:#1565C0">60</div>
    <b>structured-only</b> (person 241 to 300)<br>biomarkers only in <code>measurement</code>.
  </div>
</div>

The notes phrase the same fact a dozen different ways: *"HER2: Positive (3+ by IHC)"*,
*"Her-2/neu overexpression confirmed: 3+"*, *"HER2 amplification: DETECTED (FISH ratio 3.1)"*.
A keyword `LIKE` query breaks on that variety; a Foundation Model reads it like a clinician.

## 🏗️ Architecture & flow

```
 ┌─────────────────────────┐
 │  6 OMOP CDM tables       │   person · condition_occurrence · measurement
 │  (from shared           │   observation · drug_exposure · note
 │   foundation, 300 pts)   │
 └───────────┬─────────────┘   nb 01  (confirm + profile; read-only)
             │
     ┌───────┴────────┐
     ▼                ▼
 ┌─────────┐    ┌──────────────┐
 │ SILVER  │    │     EDA      │  nb 03  (explore the notes-only gap)
 │ pivots  │    └──────────────┘
 │ 🛠️ you  │  nb 02
 └────┬────┘
      │            ┌──────────────────────────────┐
      │            │ NLP biomarker extraction      │
      │            │  • ai_query + FMAPI 🧠 (nb 04)│
      │            │  • ClinicalBERT + MLflow(nb 05)│
      │            └───────────────┬──────────────┘
      ▼                            ▼
 ┌──────────────────────────────────────┐
 │ GOLD: unified biomarker profile +     │  nb 06  🛠️ you
 │ trial pre-screen  (data_source audit) │
 └───────────────┬──────────────────────┘
         ┌───────┴────────┐
         ▼                ▼
  ┌─────────────┐   ┌──────────────┐
  │ MLflow eval │   │ Genie space  │  nb 07 🧠 / nb 08
  │ (nb 07)     │   │ self-serve   │
  └─────────────┘   └──────────────┘            App → app/
```

## 📒 The notebooks

Run them in order. Each one is self-contained and starts by `%run ./_config`.

| # | Notebook | What it builds | Teaching moment |
|---|---|---|---|
| 01 | `01_data_foundation_omop` | Confirm & profile the 6 OMOP tables from the shared foundation | the planted cohorts + the notes-only gap |
| 02 | `02_silver_feature_pipeline` | **Silver feature views** (HER2/ER/PR pivot, therapy, demographics) | declarative SQL pipeline |
| 03 | `03_exploratory_data_analysis` | EDA, quantify & visualize the notes-only gap | the gap that justifies NLP |
| 04 | `04_nlp_biomarker_extraction` | `ai_query` + Foundation Models over `note_text` | 🧠 **ai_query** the GenAI core |
| 05 | `05_clinicalbert_mlflow_uc` | ClinicalBERT → Unity Catalog via MLflow, then a serving endpoint | 🧠 **HuggingFace + model serving** |
| 06 | `06_gold_unified_prescreen` | Gold unified profile + generic, catalog-driven pre-screen | fuse + audit + trials-as-data |
| 07 | `07_mlflow_evaluation_runs` | MLflow eval + `mlflow.genai.evaluate` traces, LLM-judge, custom metric | 🧠 **evaluation, traces, judges** |
| 08 | `08_genie_space_setup` | A Genie space for self-serve cohort questions | natural-language self-serve |
| app | `app/` | Coordinator pre-screening app | the last mile |

## ✅ Prerequisites

- A Unity Catalog-enabled workspace and permission to create a catalog/schema (your bundle's
  `client_catalog` / `client_schema`).
- **Serverless** notebook compute.
- Access to the **Foundation Model** endpoints (`databricks-claude-haiku-4-5`,
  `databricks-claude-sonnet-4-6`), used by notebooks 04 and 07.
- The 6 OMOP tables already exist. They are stood up once by the shared foundation (the
  `foundation/` bundle, which lands them in `clinops_foundation`) and this kit reads them
  read-only from your `source_schema`.
- You ran `databricks bundle deploy --target client` to sync these notebooks. See the README.
- Set the widgets below to match your bundle, then open **`01_data_foundation_omop`**.

In [0]:
%run ./_config

# ⚙️ Configuration: Clinical Trial Pre-Screening (PRE-BUILT)

<div style="background:#f4f6f9; border-left:6px solid #C8102E; padding:14px 18px; border-radius:4px; font-size:0.95em">
This is the <b>companion config notebook</b>. It is <b>pre-built; you do not edit it</b>.
Every other notebook starts with <code>%run ./_config</code> so they all share one
catalog / schema / warehouse and the same read-only OMOP source.<br>
Just set the widgets at the top of <code>00_START_HERE</code> (matching your
bundle's <code>client_catalog</code> / <code>client_schema</code> / <code>warehouse_id</code>
/ <code>source_schema</code>) and re-run.
</div>

Everything here is Unity-Catalog-scoped (no hive_metastore) and reads from widgets, no
hardcoded secrets.

ℹ️ Not creating catalog clinops_catalog (likely pre-provisioned / no CREATE CATALOG): (com.databricks.sql.managedcatalog.acl.UnauthorizedAccessException) PERMISSION_D
✅ Writing to clinops_catalog.clinops_ml
   Reading read-only OMOP source from clinops_catalog.clinops_foundation
   SQL Warehouse: 0123456789abcdef


Helpers ready: fqn(), src(), show_md(), LLM_FAST, LLM_STRONG


## ▶️ Next step

### → Open **[01_data_foundation_omop]($./01_data_foundation_omop)** to confirm and profile the OMOP data.